Based on https://github.com/micom-dev/media/blob/main/recipes/vmh_template.ipynb

qiime micom db --m-meta-file bacteria_models.tsv --p-rank species --p-threads 2 --o-metabolic-models mcnulty2013_vmh.qza

In [1]:
import pandas as pd
from cobra.io import read_sbml_model
from micom.qiime_formats import load_qiime_model_db, load_qiime_medium
from micom.workflows.db_media import complete_db_medium
from micom.workflows.db_media import check_db_medium
import qiime2 as q2

## Load diet from VMH

In [2]:
diet = pd.read_csv("media/high_fiber_diet_vmh.tsv", sep="\t")
diet.columns = ["reaction", "flux"]
annotations = pd.read_csv("media/agora_metabolites.csv")

diet = diet.rename(columns={diet.columns[0]: "reaction"})
diet["metabolite"] = diet.reaction.str.replace("^EX_", "", regex=True).str.replace("\\[e\\]|\\(e\\)", "", regex=True)
diet.loc[diet.metabolite == "4hpro", "metabolite"] = "4hpro_LT"  # fix name for hydroxyproline
diet.loc[diet.flux == 0, "flux"] = 1e-4  # bug in VMH designer where everything <1e-4 gets truncated to 0

diet

,reaction,flux,metabolite
0,EX_etoh[e],22.792196,etoh
1,EX_h2o[e],181671.336776,h2o
2,EX_caro[e],0.013132,caro
3,EX_retinol[e],1.099662,retinol
4,EX_thm[e],7.235601,thm
...,...,...,...
86,EX_lgnc[e],0.166975,lgnc
87,EX_fol[e],0.000273,fol
88,EX_strch1[e],45.113962,strch1
89,EX_i[e],0.003661,i


## Adjust for mice intestinal absorption

In [3]:
mouse = read_sbml_model("models/Mouse/iMM1865.xml")
exchanges = pd.Series([r.id for r in mouse.exchanges])
exchanges = exchanges.str.replace("__", "_").str.replace("_e$|EX_", "", regex=True)
exchanges.head()

Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


0          10fthf
1      10fthf5glu
2      10fthf6glu
3      10fthf7glu
4    11_cis_retfa
dtype: object

In [4]:
diet["dilution"] = 1.0
diet.loc[diet.metabolite.isin(exchanges), "dilution"] = 0.2
diet["flux"] = diet["flux"] * diet["dilution"] 
diet[["metabolite", "dilution"]].drop_duplicates().dilution.value_counts()

dilution
0.2    82
1.0     9
Name: count, dtype: int64

In [5]:
diet

,reaction,flux,metabolite,dilution
0,EX_etoh[e],4.558439,etoh,0.2
1,EX_h2o[e],36334.267355,h2o,0.2
2,EX_caro[e],0.002626,caro,0.2
3,EX_retinol[e],0.219932,retinol,0.2
4,EX_thm[e],1.447120,thm,0.2
...,...,...,...,...
86,EX_lgnc[e],0.033395,lgnc,0.2
87,EX_fol[e],0.000055,fol,0.2
88,EX_strch1[e],9.022792,strch1,0.2
89,EX_i[e],0.000732,i,0.2


## Adding host supplied components

In [6]:
diet.set_index("metabolite", inplace=True)

# mucin
for met in annotations.loc[annotations.metabolite.str.contains("core"), "metabolite"]:
    diet.loc[met, "flux"] = 1

# primary BAs
for met in ["gchola", "tchola"]:
    diet.loc[met, "flux"] = 1

# fiber
diet.loc["cellul", "flux"] = 10

# anaerobic
diet.loc["o2", "flux"] = 0.001

diet.reset_index(inplace=True)
diet["reaction"] = "EX_" + diet.metabolite + "(e)"
diet

,metabolite,reaction,flux,dilution
0,etoh,EX_etoh(e),4.558439,0.2
1,h2o,EX_h2o(e),36334.267355,0.2
2,caro,EX_caro(e),0.002626,0.2
3,retinol,EX_retinol(e),0.219932,0.2
4,thm,EX_thm(e),1.447120,0.2
...,...,...,...,...
102,gncore2_rl,EX_gncore2_rl(e),1.000000,NaN
103,core7,EX_core7(e),1.000000,NaN
104,gchola,EX_gchola(e),1.000000,NaN
105,tchola,EX_tchola(e),1.000000,NaN


In [7]:
diet[diet['metabolite']=='cellul']

,metabolite,reaction,flux,dilution
39,cellul,EX_cellul(e),10.0,1.0


In [8]:
skeleton = pd.merge(diet, annotations, on="metabolite", how="left")

skeleton["global_id"] = skeleton.reaction
skeleton["reaction"] = "EX_" + skeleton.metabolite + "_m"
skeleton

,metabolite,reaction,flux,dilution,name,hmdb,kegg.compound,pubchem.compound,inchi,chebi,global_id
0,etoh,EX_etoh_m,4.558439,0.2,ethanol,HMDB00108,C00469,702.0,"InChI=1S/C2H6O/c1-2-3/h3H,2H2,1H3",NaN,EX_etoh(e)
1,h2o,EX_h2o_m,36334.267355,0.2,Water,HMDB02111,C00001,962.0,InChI=1S/H2O/h1H2,NaN,EX_h2o(e)
2,caro,EX_caro_m,0.002626,0.2,beta-carotene,HMDB00561,C02094,NaN,NaN,NaN,EX_caro(e)
3,retinol,EX_retinol_m,0.219932,0.2,retinol,HMDB00305,C00473,445354.0,InChI=1S/C20H30O/c1-16(8-6-9-17(2)13-15-21)11-...,NaN,EX_retinol(e)
4,thm,EX_thm_m,1.447120,0.2,Thiamin,HMDB00235,C00378,1130.0,InChI=1S/C12H17N4OS/c1-8-11(3-4-17)18-7-16(8)6...,NaN,EX_thm(e)
...,...,...,...,...,...,...,...,...,...,...,...
102,gncore2_rl,EX_gncore2_rl_m,1.000000,NaN,"released GlcNAc-alpha-1,4-Core 2",NaN,NaN,NaN,NaN,NaN,EX_gncore2_rl(e)
103,core7,EX_core7_m,1.000000,NaN,Core 7,NaN,NaN,NaN,NaN,NaN,EX_core7(e)
104,gchola,EX_gchola_m,1.000000,NaN,glycocholate,HMDB00138,C01921,10140.0,NaN,NaN,EX_gchola(e)
105,tchola,EX_tchola_m,1.000000,NaN,taurocholate,HMDB00036,NaN,440567.0,NaN,NaN,EX_tchola(e)


## Complete medium

In [9]:
manifest, imports = complete_db_medium("mcnulty2013_vmh.qza", skeleton, growth=0.01, threads=4, max_added_import=10, weights="mass")
manifest.can_grow.value_counts()

Completing 11 species-level models on a medium with 107 components (0 strict).

Output()

Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23
Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23
Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23
Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


... (2 outputs left out)

Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Obtained growth for 11 models adding additional flux of 0.00/36699.85 on average.

can_grow
True    11
Name: count, dtype: int64

In [10]:
skeleton.drop(columns=['dilution'], inplace=True)

In [11]:
filled = imports.max()
added = filled[~filled.index.isin(skeleton.reaction)]
added.sort_values(ascending=False, inplace=True)

print(f"Added flux is {added.sum():.2f}/{filled.sum():.2f} mmol/h.")
added.sort_values(ascending=False)

Added flux is 0.01/36797.15 mmol/h.


EX_h2s_m        0.002194
EX_hxan_m       0.001751
EX_asn_L_m      0.001724
EX_cobalt2_m    0.000200
EX_so4_m        0.000200
EX_hspg_m       0.000114
EX_pheme_m      0.000031
EX_q8_m         0.000031
EX_cspg_b_m     0.000026
dtype: float64

In [12]:
added_df = filled.reset_index() 
added_df.iloc[:, 0] = added_df.iloc[:, 0].str.replace("EX_|_m$", "", regex=True)
added_df.columns = ["metabolite", "flux"]
added_df

,metabolite,flux
0,ala_L,11.812260
1,arg_L,8.134341
2,btn,0.000096
3,ca2,6.604022
4,cl,46.289002
...,...,...
76,pheme,0.000031
77,q8,0.000031
78,fol,0.000055
79,ncam,0.043275


In [13]:
completed = pd.merge(added_df, annotations, on="metabolite", how="left")
completed["reaction"] = "EX_" + completed.metabolite + "_m"
completed["global_id"] = "EX_" + completed.metabolite + "(e)"
completed = completed.loc[:,skeleton.columns]
completed = pd.concat([skeleton, completed])
completed.drop_duplicates(subset=["metabolite"], inplace=True, keep="last")
completed

,metabolite,reaction,flux,name,hmdb,kegg.compound,pubchem.compound,inchi,chebi,global_id
EX_caro_m,caro,EX_caro_m,0.002626,beta-carotene,HMDB00561,C02094,NaN,NaN,NaN,EX_caro(e)
EX_retinol_m,retinol,EX_retinol_m,0.219932,retinol,HMDB00305,C00473,445354.0,InChI=1S/C20H30O/c1-16(8-6-9-17(2)13-15-21)11-...,NaN,EX_retinol(e)
EX_adpcbl_m,adpcbl,EX_adpcbl_m,0.000003,NaN,NaN,NaN,NaN,NaN,NaN,EX_adpcbl(e)
EX_10fthf_m,10fthf,EX_10fthf_m,0.000051,10-Formyltetrahydrofolate,HMDB00972,C00234,122347.0,NaN,NaN,EX_10fthf(e)
EX_5mthf_m,5mthf,EX_5mthf_m,0.000052,5-Methyltetrahydrofolate,HMDB01396,C00440,439234.0,NaN,NaN,EX_5mthf(e)
...,...,...,...,...,...,...,...,...,...,...
76,pheme,EX_pheme_m,0.000031,Protoheme,HMDB03178,C00032,NaN,NaN,NaN,EX_pheme(e)
77,q8,EX_q8_m,0.000031,Ubiquinone-8,NaN,NaN,NaN,NaN,NaN,EX_q8(e)
78,fol,EX_fol_m,0.000055,Folate,HMDB00121,C00504,6037.0,NaN,NaN,EX_fol(e)
79,ncam,EX_ncam_m,0.043275,Nicotinamide,HMDB01406,C00153,936.0,NaN,NaN,EX_ncam(e)


## Validate medium

In [14]:
check = check_db_medium("mcnulty2013_vmh.qza", medium=completed, threads=4)

Checking 11 species-level models on a medium with 116 components.

Output()

Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23
Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23
Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23
Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23
Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23
Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-23


In [15]:
check.growth_rate.describe()

count    11.000000
mean      0.018749
std       0.008405
min       0.010000
25%       0.010000
50%       0.025641
75%       0.025641
max       0.028037
Name: growth_rate, dtype: float64

In [16]:
arti = q2.Artifact.import_data("MicomMedium[Global]", completed)

In [17]:
arti.save("media/vmh_high_fiber_.qza")

'media/vmh_high_fiber_.qza'

In [18]:
medium = load_qiime_medium("media/vmh_high_fiber_.qza")
medium.sort_values("flux", ascending=False, inplace=True)
medium.to_csv("media/vmh_high_fiber_.tsv", sep="\t", index=False)
medium

,metabolite,reaction,flux,name,hmdb,kegg.compound,pubchem.compound,inchi,chebi,global_id
reaction,,,,,,,,,,
EX_h2o_m,h2o,EX_h2o_m,36334.267355,Water,HMDB02111,C00001,962.0,InChI=1S/H2O/h1H2,NaN,EX_h2o(e)
EX_cl_m,cl,EX_cl_m,46.289002,Chloride,HMDB00492,C00698,312.0,InChI=1S/ClH/h1H/p-1,NaN,EX_cl(e)
EX_fru_m,fru,EX_fru_m,31.465459,D-Fructose,HMDB00660,C02336,439709.0,NaN,NaN,EX_fru(e)
EX_mg2_m,mg2,EX_mg2_m,31.155318,magnesium,NaN,NaN,NaN,NaN,NaN,EX_mg2(e)
EX_na1_m,na1,EX_na1_m,29.582376,Sodium,HMDB00588,C01330,923.0,InChI=1S/Na/q+1,NaN,EX_na1(e)
...,...,...,...,...,...,...,...,...,...,...
EX_cspg_b_m,cspg_b,EX_cspg_b_m,0.000026,chondroitin sulfate B / dermatan sulfate (IdoA...,NaN,NaN,NaN,NaN,NaN,EX_cspg_b(e)
EX_adrn_m,adrn,EX_adrn_m,0.000020,NaN,NaN,NaN,NaN,NaN,NaN,EX_adrn(e)
EX_strdnc_m,strdnc,EX_strdnc_m,0.000020,NaN,NaN,NaN,NaN,NaN,NaN,EX_strdnc(e)
